# Math 638 Project 2 - Homework 6.2

## Attention, Glyphs, and Contextual Decipherment

This report presents Homework 6.2 as a formal deliverables-style analysis based on the final submitted folder.

The project returns to the artificial Marcus Aurelius decipherment experiment from Homework 5.1 and asks whether self-attention provides a meaningful advantage when decoding glyphs from surrounding context.

The central modeling change is:

**fixed-context non-attention decoder -> centered self-attention decoder.**

The corpus, tokenization, source vocabulary, glyph vocabulary, hidden bijection, train/validation/test split, nested anchor sets, and Republic-informed pseudo-label construction are all carried over from Homework 5.1.

The final experiment therefore isolates decoder behavior as much as possible while preserving the original information restriction.

## Executive Summary

Homework 6.2 reuses the complete 12-book *Meditations* corpus from Homework 5.1 with

$$
77{,}260
$$

written-syllable token occurrences and

$$
K=3052
$$

source syllable types and artificial glyph types.

The original split is retained:

- 54,082 training glyphs;
- 11,589 validation glyphs;
- 11,589 test glyphs.

Nested revealed-anchor sets contain 153, 305, 610, and 1,221 glyph types at approximately 5%, 10%, 20%, and 40%.

The selected attention architecture has

$$
d=96,\qquad m=4,\qquad h=2,\qquad L=1,
$$

meaning a 96-dimensional embedding, centered context radius 4, two attention heads, and one self-attention layer. The window length is

$$
2m+1=9,
$$

and the model has 664,684 trainable parameters. Architecture selection used validation pseudo-label negative log-likelihood only.

### Main bijective-cipher result

In the original one-to-one cipher, attention and the matched MLP baseline perform almost identically.

At 20% anchors, whole-text test accuracy is

$$
0.26798
$$

for attention and

$$
0.26728
$$

for the MLP.

More importantly, unanchored accuracy is **exactly the same** for the two decoders:

$$
0.09356.
$$

This pattern persists across every anchor level. The attention model therefore does not solve the fundamental problem of assigning names to genuinely unrevealed syllables.

### Controlled ambiguous-glyph result

Forty glyph pairs are merged after the Homework 5.1 labels are frozen, producing a controlled many-to-one observed-symbol experiment.

Attention shows a modest advantage at 5% and 10% anchors, but the advantage is not stable:

- at 5% anchors, ambiguous-occurrence accuracy improves by about 0.97 percentage points;
- at 10%, by about 1.19 points;
- at 20%, attention is worse by about 1.78 points;
- at 40%, attention is slightly worse.

Thus the data do **not** support a general statement that attention is more accurate than a matched MLP whenever glyph identity is ambiguous.

### Strong context-sensitivity result

The intervention experiment gives a more important result.

For ambiguous centers, replacing the highest-attended non-center symbol produces a mean absolute true-label probability change of

$$
0.19248,
$$

compared with only

$$
0.00203
$$

for the lowest-attended control.

The argmax prediction changes in

$$
37.04\%
$$

of top-attention replacements but only

$$
0.82\%
$$

of low-attention replacements.

Context permutation changes the ambiguous prediction on

$$
38.07\%
$$

of sampled occurrences.

Therefore the selected attention model has learned **real context-sensitive behavior for ambiguous observed symbols**, even though this behavior does not consistently produce higher total decoding accuracy than the MLP.

### Failure-mode result

Increasing revealed anchors has a much larger effect on hidden-label recovery than merely increasing text volume. Unanchored accuracy rises from about

$$
0.0362
$$

at 5% anchors to

$$
0.1571
$$

at 40% anchors.

By contrast, increasing training text from 25% to 100% changes whole-text accuracy only from about 0.2625 to 0.2680, while unanchored accuracy remains 0.09356.

The final conclusion is therefore:

> **Attention learns meaningful contextual sensitivity when the observed symbol is genuinely ambiguous, but it does not remove the identifiability barrier for syllable names that were never revealed by anchors or the permitted Homework 5.1 supervision.**

## Deliverables Overview

| Task | Deliverable | Status |
| --- | --- | --- |
| 1 | Reuse of HW5.1 corpus, tokenization, glyph vocabulary, and hidden map | Complete |
| 2 | Same split, nested anchors, and allowed-information regime | Complete |
| 3 | Centered self-attention decoder | Complete |
| 4 | Training under the HW5.1 information restriction | Complete |
| 5 | Attention versus matched non-attention decoder | Complete |
| 6 | Embedding/context/head/layer architecture study | Complete |
| 7 | Controlled one-to-one versus many-to-one glyph experiment | Complete |
| 8 | Attention-weight visualization and aggregate context analysis | Complete |
| 9 | LLM interpretation plus removal/permutation interventions | Complete |
| 10 | Failure-mode analysis | Complete |

# 1. Reuse of the Homework 5.1 Decipherment System

Homework 6.2 keeps the underlying decipherment problem fixed.

The carried-over source is the complete public-domain English translation of Marcus Aurelius' *Meditations* by Meric Casaubon, Project Gutenberg eBook #2680.

The frozen Homework 5.1 configuration is:

| Component | Reused Homework 6.2 value |
| --- | --- |
| Corpus | Complete 12-book *Meditations* |
| Written-syllable occurrences | 77,260 |
| Source vocabulary size | 3,052 |
| Glyph vocabulary size | 3,052 |
| Tokenizer | Pyphen 0.18.1, `en_US` |
| Source IDs | Same as Homework 5.1 |
| Glyph IDs | Same as Homework 5.1 |
| Hidden map | Same seeded bijection $\Phi$ |
| New hidden map generated | No |

The same normalization and written-syllable reconstruction rules are retained.

The experimental principle is:

**Keep the cipher fixed and change the decoder.**

The complete hidden inverse map exists only as an evaluation answer key. Its unrevealed entries are not available during fitting or architecture selection.

# 2. Frozen Split, Nested Anchors, and Information Regime

The original contiguous 70% / 15% / 15% split is reused exactly.

| Split | Glyph occurrences | Fraction | Distinct glyphs |
| --- | --- | --- | --- |
| Train | 54,082 | 70.0% | 2,671 |
| Validation | 11,589 | 15.0% | 1,305 |
| Test | 11,589 | 15.0% | 1,365 |

The nested anchor sets are also reused without resampling:

| Anchor level | Revealed glyph types | Actual percentage |
| --- | --- | --- |
| 5% | 153 | 5.013% |
| 10% | 305 | 9.993% |
| 20% | 610 | 19.987% |
| 40% | 1,221 | 40.007% |

The anchor condition satisfies

$$
A_5
\subseteq
A_{10}
\subseteq
A_{20}
\subseteq
A_{40}.
$$

The permitted fitting information is:

- training glyph sequences;
- revealed anchors;
- the frozen Republic-informed information from Homework 5.1;
- the carried-over pseudo-label maps;
- validation pseudo-labels for model selection.

The following are forbidden until final evaluation:

- unrevealed *Meditations* plaintext;
- unrevealed entries of the true inverse map;
- held-out test labels.

A common set of 11,557 validation centers and 11,557 test centers is frozen so models with different context radii can be compared on identical positions.


# 3. Centered Self-Attention Decoder

For context radius $m$, the observed input is

$$
g_{i-m},\ldots,g_i,\ldots,g_{i+m}.
$$

The target is the source syllable identity at the center position:

$$
P_\theta
(
t_i
\mid
g_{i-m},\ldots,g_i,\ldots,g_{i+m}
).
$$

Because the task is contextual decipherment rather than next-token prediction, no causal mask is used. Both left and right context are permitted.

For embedded window matrix $H$,

$$
Q=HW_Q,\qquad
K=HW_K,\qquad
V=HW_V,
$$

and each self-attention head computes

$$
A
=
softmax
(
\frac{QK^\top}{\sqrt{d_k}}
).
$$

The representation at the **center position** is passed to the source-syllable classifier.

The selected architecture contains:

- embedding dimension: 96;
- context radius: 4;
- window length: 9;
- attention heads: 2;
- self-attention layers: 1;
- trainable parameters: 664,684.

The matched non-attention baseline uses the same 9-glyph context, the same 96-dimensional embeddings, the same pseudo-labels, and a 96-unit MLP hidden layer with 672,076 trainable parameters.

# 4. Training Procedure


At anchor level $a$, the training target is the carried-over permitted Homework 5.1 pseudo-label

$$
\hat{\pi}_a(g_i),
$$

not the hidden true plaintext label.

The loss is

$$
L_a(\theta)
=
-\frac{1}{|I|}
\sum_{i\in I}
\log
P_\theta
(
\hat{\pi}_a(g_i)
\mid
g_{i-m},\ldots,g_{i+m}
).
$$

Gradient updates use the training split.

Validation pseudo-label negative log-likelihood determines early stopping and architecture selection.

The test split and true inverse map are opened only after all architecture and checkpoint decisions are fixed.

The final trained models are:


| Anchors | Model | Radius | Window | Embedding | Heads | Layers | Parameters | Best epoch | Validation NLL | Validation accuracy |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 5% | Attention | 4 | 9 | 96 | 2 | 1 | 664,684 | 4 | 0.194453 | 0.980272 |
| 5% | MLP baseline | 4 | 9 | 96 | 0 | 0 | 672,076 | 8 | 0.282785 | 0.978541 |
| 10% | Attention | 4 | 9 | 96 | 2 | 1 | 664,684 | 4 | 0.196258 | 0.980272 |
| 10% | MLP baseline | 4 | 9 | 96 | 0 | 0 | 672,076 | 8 | 0.288344 | 0.978801 |
| 20% | Attention | 4 | 9 | 96 | 2 | 1 | 664,684 | 4 | 0.196666 | 0.980272 |
| 20% | MLP baseline | 4 | 9 | 96 | 0 | 0 | 672,076 | 7 | 0.284612 | 0.979320 |
| 40% | Attention | 4 | 9 | 96 | 2 | 1 | 664,684 | 4 | 0.195135 | 0.980272 |
| 40% | MLP baseline | 4 | 9 | 96 | 0 | 0 | 672,076 | 8 | 0.284003 | 0.978282 |

# 5. Bijective Cipher: Attention Versus Matched MLP

| Anchors | Model | Whole-text accuracy | Anchored accuracy | Unanchored accuracy | All-K map recovery | Observed-type map recovery | Unanchored map recovery |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 5% | Attention | 0.0893 | 0.9876 | 0.0362 | 0.0242 | 0.0542 | 0.0021 |
| 5% | MLP | 0.0892 | 0.9860 | 0.0362 | 0.0239 | 0.0535 | 0.0021 |
| 10% | Attention | 0.1638 | 0.9827 | 0.0777 | 0.0452 | 0.1011 | 0.0058 |
| 10% | MLP | 0.1636 | 0.9809 | 0.0777 | 0.0446 | 0.0996 | 0.0058 |
| 20% | Attention | 0.2680 | 0.9819 | 0.0936 | 0.0793 | 0.1773 | 0.0029 |
| 20% | MLP | 0.2673 | 0.9784 | 0.0936 | 0.0777 | 0.1736 | 0.0029 |
| 40% | Attention | 0.4543 | 0.9799 | 0.1571 | 0.1589 | 0.3553 | 0.0087 |
| 40% | MLP | 0.4529 | 0.9760 | 0.1571 | 0.1547 | 0.3458 | 0.0087 |

The dominant effect is the anchor percentage, not the decoder family.

At 5% anchors, attention achieves only

$$
3.62\%
$$

unanchored occurrence accuracy.

At 40% anchors, that rises to

$$
15.71\%.
$$

However, the attention and MLP unanchored accuracies are **identical at every anchor level**.

The small whole-text advantage of attention comes almost entirely from slightly better prediction on already anchored glyph occurrences.

The same conclusion appears in inverse-map recovery. Attention's all-$K$ recovery is slightly higher than the MLP's, but recovery on genuinely unanchored map entries remains extremely low.

Therefore the one-to-one experiment does not show that attention overcomes the identifiability problem for unseen syllable names.

![Whole-text accuracy versus anchors](../result%20images/hw6_2/task5_overall_accuracy.png)

![Unanchored accuracy versus anchors](../result%20images/hw6_2/task5_unanchored_accuracy.png)

![Inverse-map recovery versus anchors](../result%20images/hw6_2/task5_map_recovery.png)


# 6. Architecture Study


The architecture sweep varies all four quantities required by the assignment:

- embedding dimension;
- context radius;
- attention heads;
- attention layers.

The 20% anchor condition is used for validation-only architecture selection.


| Experiment | Embedding | Radius | Heads | Layers | Parameters | Validation NLL | Validation accuracy |
| --- | --- | --- | --- | --- | --- | --- | --- |
| embed_96 | 96 | 4 | 2 | 1 | 664,684 | 0.196854 | 0.980272 |
| heads_1 | 64 | 4 | 1 | 1 | 427,756 | 0.204591 | 0.980272 |
| radius_8 | 64 | 8 | 2 | 1 | 428,268 | 0.205718 | 0.980272 |
| radius_2 | 64 | 2 | 2 | 1 | 427,500 | 0.205740 | 0.980272 |
| radius_16 | 64 | 16 | 2 | 1 | 429,292 | 0.206489 | 0.980272 |
| reference | 64 | 4 | 2 | 1 | 427,756 | 0.206564 | 0.980272 |
| layers_2 | 64 | 4 | 2 | 2 | 461,228 | 0.206750 | 0.980272 |
| heads_4 | 64 | 4 | 4 | 1 | 427,756 | 0.207798 | 0.980272 |
| embed_32 | 32 | 4 | 2 | 1 | 207,212 | 0.238195 | 0.980272 |

The selected model is the 96-dimensional embedding configuration:

$$
d=96,\qquad
m=4,\qquad
h=2,\qquad
L=1,
$$

with validation NLL

$$
0.196854.
$$

All tested architectures have almost identical validation pseudo-label accuracy, approximately 0.98027. The selection is therefore driven by NLL rather than accuracy.

The results do not show a monotone relationship between context size, number of heads, depth, or parameter count and validation performance. A larger embedding helps relative to the 64-dimensional reference, but doubling the number of layers or increasing context radius does not consistently improve the result.

![Architecture validation NLL](../result%20images/hw6_2/task6_architecture_validation_nll.png)


# 7. Controlled Many-to-One Ambiguous Glyph Experiment

To create genuine context-dependent ambiguity, 40 disjoint pairs of original glyphs are merged.

Thus 80 original glyph types participate in the controlled many-to-one experiment.

The pair construction respects the nested-anchor signature

$$
s(g)
=
(1_{g\in A_5},
1_{g\in A_{10}},
1_{g\in A_{20}},
1_{g\in A_{40}}).
$$

Only glyphs with identical signatures are merged.

This guarantees that a merged group is fully revealed or fully unrevealed at each anchor level.

The labels are fixed on the original sequence **before** merging. The decoder sees only the merged observed-symbol ID and does not receive the original pre-merge identity.

## 7.1 Ambiguous-decoding results

| Anchors | Model | Whole-text accuracy | Unanchored accuracy | Ambiguous accuracy | Revealed ambiguous accuracy | Unrevealed ambiguous accuracy | Exact relation recovery | Mean relation Jaccard |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 5% | Attention | 0.0788 | 0.0277 | 0.1197 | 0.9406 | 0.0435 | 0.200 | 0.225 |
| 5% | MLP | 0.0738 | 0.0227 | 0.1099 | 0.9406 | 0.0329 | 0.200 | 0.225 |
| 10% | Attention | 0.1500 | 0.0711 | 0.1784 | 0.8910 | 0.0506 | 0.400 | 0.425 |
| 10% | MLP | 0.1433 | 0.0652 | 0.1665 | 0.8863 | 0.0374 | 0.400 | 0.425 |
| 20% | Attention | 0.2371 | 0.0734 | 0.3830 | 0.9001 | 0.1544 | 0.625 | 0.642 |
| 20% | MLP | 0.2434 | 0.0815 | 0.4008 | 0.9142 | 0.1739 | 0.625 | 0.642 |
| 40% | Attention | 0.4210 | 0.1430 | 0.5581 | 0.8971 | 0.2320 | 0.800 | 0.833 |
| 40% | MLP | 0.4174 | 0.1482 | 0.5608 | 0.8886 | 0.2454 | 0.800 | 0.833 |

For an ambiguous observed symbol $z$, the true object is no longer an inverse function. It is a relation

$$
R(z)\subseteq\mathcal T.
$$

Because the controlled merge pairs contain two original glyphs,

$$
|R(z)|=2.
$$

The relation-recovery score therefore asks whether the two highest aggregated source-syllable candidates equal the true two-element relation.

Relation recovery increases strongly with anchor percentage:

$$
20\%,\quad
40\%,\quad
62.5\%,\quad
80\%
$$

exact recovery at 5%, 10%, 20%, and 40% anchors.

Notably, attention and the MLP obtain the same relation-recovery scores at every anchor level.


## 7.2 Does attention help more when the observed glyph is ambiguous?

| Anchors | Bijective whole-text advantage | Ambiguous whole-text advantage | Ambiguous unanchored advantage | Ambiguous-occurrence advantage |
| --- | --- | --- | --- | --- |
| 5% | +0.01 pp | +0.50 pp | +0.49 pp | +0.97 pp |
| 10% | +0.02 pp | +0.67 pp | +0.59 pp | +1.19 pp |
| 20% | +0.07 pp | -0.63 pp | -0.81 pp | -1.78 pp |
| 40% | +0.14 pp | +0.36 pp | -0.51 pp | -0.27 pp |

The answer is **not consistently**.

At low anchor levels, attention has a small positive advantage in the ambiguous experiment.

At 20% anchors, however, the matched MLP is better:

$$
Acc_{ambiguous}
=
0.38295
$$

for attention versus

$$
0.40079
$$

for the MLP.

At 40% anchors, attention is again slightly worse on ambiguous occurrences.

Therefore the experiment does not support the broad claim that **ambiguity implies attention is always more accurate**.

The more defensible question is whether attention has learned **context-sensitive behavior**, which is tested in Tasks 8 and 9.


# 8. Attention-Weight Analysis


Two held-out examples are selected reproducibly using the same merged observed symbol. The two contexts correspond to different underlying true syllables, and the attention model decodes both correctly.


| Example | Position | Observed symbol | True source token | Predicted source token | Confidence |
| --- | --- | --- | --- | --- | --- |
| 1 | 4411 | 2972 | 76 | 76 | 0.9847 |
| 2 | 4010 | 2972 | 1697 | 1697 | 0.9063 |

These examples show that the same observed merged symbol can receive different predictions in different contexts.

Aggregate analysis is more important than the two selected figures:


| Center type | Prediction | Occurrences | Mean expected |attention distance| | Mean attention entropy |
| --- | --- | --- | --- | --- |
| Unambiguous | Incorrect | 5393 | 0.7764 | 1.2257 |
| Unambiguous | Correct | 615 | 0.9484 | 1.3063 |
| Ambiguous | Incorrect | 3424 | 1.1204 | 1.0751 |
| Ambiguous | Correct | 2125 | 1.1573 | 0.9668 |

Ambiguous centers attend farther from the center on average than unambiguous centers.

For example, correct ambiguous cases have mean expected absolute attention distance approximately

$$
1.157,
$$

compared with approximately

$$
0.948
$$

for correct unambiguous cases.

This is evidence of an association between ambiguity and broader contextual use.

It is **not** yet causal evidence that the largest attention weights identify the positions responsible for the model's decision.

![Attention matrix - example 1](../result%20images/hw6_2/task8_attention_matrix_example_1.png)

![Center-query profile - example 1](../result%20images/hw6_2/task8_attention_profile_example_1.png)

![Attention matrix - example 2](../result%20images/hw6_2/task8_attention_matrix_example_2.png)

![Center-query profile - example 2](../result%20images/hw6_2/task8_attention_profile_example_2.png)


# 9. LLM Critic and Context Interventions


The LLM is first shown only the fixed decoding results, attention summaries, and selected attention plots.

It is **not** shown the intervention outcomes.

It is then asked to make explicit predictions about:

- replacing the highest-attended non-center symbol;
- replacing the lowest-attended non-center symbol;
- permuting all non-center context symbols;
- whether effects should be stronger for ambiguous than unambiguous centers.

This makes the LLM interpretation testable rather than purely descriptive.

## 9.1 Exact LLM critic prompt

```text
You are the critic for a Math 638 contextual-decipherment experiment.

The model is a centered self-attention decoder for an artificial glyph cipher. Some original glyphs have been deliberately merged into ambiguous observed symbols.

You are given fixed decoding results, attention summaries, and selected attention plots. You have NOT been given the context-intervention results.

Answer the following questions using only the supplied evidence.

1. What do the selected attention plots appear to emphasize when the same ambiguous observed glyph is decoded differently in different contexts?
2. Does the aggregate evidence suggest that ambiguous occurrences use context differently from unambiguous occurrences?
3. Does attention appear more helpful than the matched non-attention baseline in the ambiguous experiment than in the bijective experiment?
4. Which parts of the attention interpretation are strongly supported, and which are only plausible descriptions?
5. Make a specific prediction: if the highest-attended non-center context symbol is replaced, should the model's probability of the true center label change more than if the lowest-attended non-center symbol is replaced?
6. Make a specific prediction about permuting all non-center context symbols while leaving the center glyph fixed.
7. Would your predicted intervention effect be stronger for ambiguous occurrences than unambiguous occurrences? State your expectation and why.

Important constraints:
- Large attention weight is not proof of causal importance.
- Do not claim the model discovered linguistic rules unless the evidence tests that claim.
- Do not infer unavailable punctuation, word-boundary, or repeated-phrase features unless they were actually supplied as model inputs.
- Distinguish resolving context-dependent ambiguity from naming genuinely unrevealed syllables.

Return:
A. Evidence-supported interpretation.
B. Explicit intervention predictions.
C. Claims that would exceed the current evidence.

FIXED PRE-INTERVENTION EVIDENCE:

BIJECTIVE DECODER RESULTS:
 anchor_level        model  evaluated_positions  overall_accuracy  anchored_occurrences  anchored_accuracy  unanchored_occurrences  unanchored_accuracy  all_K_map_recovery  observed_type_map_recovery  observed_test_glyph_types  unanchored_map_recovery
            5    attention                11557          0.089297                   645           0.987597                   10912             0.036199            0.024246                    0.054212                       1365                 0.002070
            5 mlp_baseline                11557          0.089210                   645           0.986047                   10912             0.036199            0.023919                    0.053480                       1365                 0.002070
           10    attention                11557          0.163797                  1100           0.982727                   10457             0.077651            0.045216                    0.101099                       1365                 0.005825
           10 mlp_baseline                11557          0.163624                  1100           0.980909                   10457             0.077651            0.044561                    0.099634                       1365                 0.005825
           20    attention                11557          0.267976                  2269           0.981930                    9288             0.093562            0.079292                    0.177289                       1365                 0.002867
           20 mlp_baseline                11557          0.267284                  2269           0.978405                    9288             0.093562            0.077654                    0.173626                       1365                 0.002867
           40    attention                11557          0.454270                  4174           0.979875                    7383             0.157118            0.158912                    0.355311                       1365                 0.008738
           40 mlp_baseline                11557          0.452886                  4174           0.976042                    7383             0.157118            0.154653                    0.345788                       1365                 0.008738

AMBIGUOUS DECODER RESULTS:
 anchor_level        model  evaluated_positions  whole_text_accuracy  unanchored_occurrences  unanchored_accuracy  ambiguous_occurrences  ambiguous_accuracy  revealed_ambiguous_occurrences  revealed_ambiguous_accuracy  unrevealed_ambiguous_occurrences  unrevealed_ambiguous_accuracy  relation_groups_total  relation_groups_observed  relation_exact_all_groups  relation_exact_observed_groups  relation_mean_jaccard_observed
            5    attention                11557             0.078827                   10912             0.027676                   5549            0.119661                             471                     0.940552                              5078                       0.043521                     40                        40                      0.200                           0.200                        0.225000
            5 mlp_baseline                11557             0.073808                   10912             0.022727                   5549            0.109930                             471                     0.940552                              5078                       0.032887                     40                        40                      0.200                           0.200                        0.225000
           10    attention                11557             0.149952                   10457             0.071149                   5549            0.178411                             844                     0.890995                              4705                       0.050584                     40                        40                      0.400                           0.400                        0.425000
           10 mlp_baseline                11557             0.143290                   10457             0.065219                   5549            0.166516                             844                     0.886256                              4705                       0.037407                     40                        40                      0.400                           0.400                        0.425000
           20    attention                11557             0.237086                    9288             0.073428                   5549            0.382952                            1701                     0.900059                              3848                       0.154366                     40                        40                      0.625                           0.625                        0.641667
           20 mlp_baseline                11557             0.243402                    9288             0.081503                   5549            0.400793                            1701                     0.914168                              3848                       0.173857                     40                        40                      0.625                           0.625                        0.641667
           40    attention                11557             0.421044                    7383             0.143031                   5549            0.558119                            2721                     0.897097                              2828                       0.231966                     40                        40                      0.800                           0.800                        0.833333
           40 mlp_baseline                11557             0.417409                    7383             0.148178                   5549            0.560822                            2721                     0.888644                              2828                       0.245403                     40                        40                      0.800                           0.800                        0.833333

ATTENTION ADVANTAGE COMPARISON:
 anchor_level  bijective_whole_text_attention_advantage  bijective_unanchored_attention_advantage  ambiguous_whole_text_attention_advantage  ambiguous_unanchored_attention_advantage  ambiguous_occurrence_attention_advantage
            5                                  0.000087                                       0.0                                  0.005019                                  0.004949                                  0.009731
           10                                  0.000173                                       0.0                                  0.006663                                  0.005929                                  0.011894
           20                                  0.000692                                       0.0                                 -0.006317                                 -0.008075                                 -0.017841
           40                                  0.001384                                       0.0                                  0.003634                                 -0.005147                                 -0.002703

SELECTED ATTENTION EXAMPLES:
 example  position  observed_symbol_index  true_source_token_id  predicted_source_token_id  confidence                                                                                             selection_reason
       1      4411                   2972                    76                         76    0.984728 same ambiguous observed symbol; different underlying values; different predictions; both predictions correct
       2      4010                   2972                  1697                       1697    0.906299 same ambiguous observed symbol; different underlying values; different predictions; both predictions correct

AGGREGATE ATTENTION SUMMARY:
,,expected_absolute_attention_distance,expected_absolute_attention_distance,expected_absolute_attention_distance,expected_absolute_attention_distance,attention_entropy,attention_entropy,attention_entropy,attention_entropy
,,count,mean,median,std,count,mean,median,std
ambiguous_occurrence,correct,,,,,,,,
False,False,5393,0.776351345142707,0.7352719185873866,0.4156693119607916,5393,1.2257105208575538,1.2788769006729126,0.3853031573964079
False,True,615,0.9483625609786283,0.8674786149058491,0.5518586815589822,615,1.3062823233565664,1.323941946029663,0.43629727662668966
True,False,3424,1.1204313856090367,1.043636744194373,0.39380661198272104,3424,1.075085334277578,1.070400893688202,0.3267416573472021
True,True,2125,1.1573436012413023,1.0591805014410056,0.42488454117710883,2125,0.9668262520053369,0.9432929754257202,0.41295630302823266


ATTENTION FIGURES TO INSPECT:
task8_attention_matrix_example_1.png
task8_attention_profile_example_1.png
task8_attention_matrix_example_2.png
task8_attention_profile_example_2.png
```

## 9.2 Verbatim LLM response

```text
A. Evidence-supported interpretation

1. The two selected examples show that the same merged observed symbol can be decoded as two different source tokens when its surrounding context changes. In example 1, the center-query attention is concentrated mainly at relative positions -1 and +1. In example 2, it is concentrated mainly at -3 and +1. This is evidence that the decoder conditions its decision on different surrounding positions in different contexts. It is not, by itself, evidence that those positions have a particular linguistic meaning.

2. The aggregate attention statistics suggest that ambiguous occurrences use context differently from unambiguous occurrences. Ambiguous occurrences have larger expected absolute attention distance: about 1.12-1.16 tokens for incorrect/correct ambiguous cases, compared with about 0.78-0.95 for unambiguous cases. Correct ambiguous cases also have somewhat lower attention entropy than incorrect ambiguous cases (about 0.97 versus 1.08), suggesting that successful ambiguous decisions are sometimes associated with more concentrated attention. These are descriptive associations, not causal conclusions.

3. Attention is not uniformly more helpful than the matched MLP baseline in the ambiguous experiment. In the bijective experiment the attention advantage is essentially zero on unanchored occurrences and only very small on whole-text accuracy. In the ambiguous experiment attention has a modest positive advantage at 5% and 10% anchors, but the advantage becomes negative for several metrics at 20% and 40% anchors. Therefore the evidence supports, at most, the narrower claim that attention can help in some low-anchor ambiguous conditions; it does not support a general claim that attention is superior whenever glyphs are ambiguous.

4. Strongly supported claims are: (i) the learned attention distribution changes with context, (ii) ambiguous occurrences tend to place attention farther from the center than unambiguous occurrences, and (iii) the same ambiguous observed symbol can receive different correct predictions in different contexts. Only plausible, not yet causally established, is the claim that the largest attention weights identify the context tokens that are responsible for the prediction.

B. Explicit intervention predictions

1. Replacing the highest-attended non-center symbol should, on average, change the model output more than replacing the lowest-attended non-center symbol. I would test this using both the absolute change in the true-label probability and the fraction of examples whose argmax prediction changes.

2. Permuting all non-center context symbols while leaving the center symbol fixed should usually be more disruptive than the low-attention replacement because it destroys the learned positional arrangement of the surrounding context. Its effect may be comparable to, or larger than, replacing a single top-attended position.

3. I expect both interventions to have a substantially stronger effect for ambiguous occurrences than for unambiguous occurrences. For an unambiguous center symbol, the center identity itself can often determine the pseudo-label. For a merged ambiguous center, surrounding context is required to distinguish which underlying source value is appropriate.

C. Claims that would exceed the current evidence

The current evidence does not justify saying that the model has discovered grammatical rules, word boundaries, punctuation structure, repeated phrases, or any other linguistic feature that was not explicitly supplied and tested. It also does not justify treating an attention weight as a causal attribution before intervention. Finally, contextual disambiguation of a merged symbol is different from recovering the English name of an unrevealed syllable: attention can use context to choose among values represented in the supervision without solving the identifiability problem for genuinely unrevealed names.
```


## 9.3 Intervention results

| Center type | Intervention | N | Mean signed true-probability drop | Mean absolute probability change | Prediction changed |
| --- | --- | --- | --- | --- | --- |
| Unambiguous | low attention replacement | 514 | -0.000002 | 0.000011 | 0.00% |
| Unambiguous | top attention replacement | 514 | -0.000123 | 0.000398 | 1.17% |
| Unambiguous | noncenter permutation | 514 | 0.000229 | 0.000464 | 1.17% |
| Ambiguous | low attention replacement | 486 | -0.000283 | 0.002027 | 0.82% |
| Ambiguous | top attention replacement | 486 | 0.078015 | 0.192482 | 37.04% |
| Ambiguous | noncenter permutation | 486 | 0.091524 | 0.174213 | 38.07% |

The intervention results strongly support the LLM's narrow testable predictions.

For **ambiguous** centers:

- high-attention replacement changes the true-label probability by about 0.1925 in absolute value;
- low-attention replacement changes it by only about 0.0020;
- high-attention replacement changes the argmax prediction in 37.04% of cases;
- low-attention replacement changes it in only 0.82%;
- non-center permutation changes the argmax prediction in 38.07%.

For **unambiguous** centers, all effects are much smaller.

The top-attention intervention is therefore about

$$
\frac{0.19248}{0.00203}\approx95
$$

times larger than the low-attention control in mean absolute probability change for ambiguous centers.

Compared with unambiguous top-attention replacement, the ambiguous effect is approximately

$$
\frac{0.19248}{0.0003976}\approx484
$$

times larger.

This is substantially stronger evidence than visualization alone: the model is genuinely using contextual information when the observed center symbol is ambiguous.

However, replacing a symbol can create an unusual context, and the network contains embeddings, residual pathways, feed-forward layers, and nonlinear interactions. Thus attention weights should still be described as **predictively informative under the tested interventions**, not as a complete causal decomposition.

![Context intervention sensitivity](../result%20images/hw6_2/task9_intervention_sensitivity.png)


# 10. Human Critique of the LLM Response

The intervention results support the LLM's main **testable** prediction, but they also show why the interpretation must remain narrow.

## 1. Did high-attention replacement matter more than the low-attention control?

Yes, very strongly for ambiguous occurrences. For ambiguous centers, replacing the highest-attended non-center symbol produced a mean absolute change in true-label probability of

$$
0.19248,
$$

whereas replacing the lowest-attended non-center symbol produced only

$$
0.00203.
$$

The corresponding argmax prediction-change rates were

The corresponding argmax prediction-change rates were **37.04% versus 0.82%**.

Thus the high-attention intervention was about two orders of magnitude larger in probability effect. This supports the claim that, **for this trained model and these ambiguous examples, high-attention context positions are associated with much greater output sensitivity than low-attention positions**.

For unambiguous centers the same ordering holds, but both effects are tiny: the mean absolute true-probability change is approximately

$$
3.98\times 10^{-4}
$$

for top-attention replacement and

$$
1.12\times 10^{-5}
$$

for low-attention replacement. Only about 1.17% of unambiguous predictions change under the top-attention replacement and none change under the low-attention replacement.

## 2. Did context permutation have the predicted effect?

Yes. For ambiguous centers, permuting all non-center positions gives a mean absolute change in true-label probability of

$$
0.17421
$$

and changes the argmax prediction on

$$
38.07\%
$$

of the sampled occurrences. This is vastly larger than the low-attention control and is comparable to the top-attention replacement. The mean signed drop in true-label probability is also positive and substantial:

$$
0.09152
$$

for permutation and

$$
0.07801
$$

for top-attention replacement.

For unambiguous occurrences, permutation again has only a very small effect. Therefore the model is genuinely sensitive to the surrounding arrangement primarily when the center symbol is ambiguous.

## 3. Were ambiguous occurrences more context-sensitive?

Yes. This is the clearest aggregate result. Under top-attention replacement, the mean absolute probability change is about

$$
\frac{0.19248}{0.0003976}\approx 484
$$

times larger for ambiguous than unambiguous occurrences. Prediction changes occur for 37.04% of ambiguous cases but only 1.17% of unambiguous cases. The permutation experiment gives the same qualitative conclusion.

This supports the assignment's intended distinction: context can help distinguish different **occurrences of a many-to-one observed symbol**.

## 4. Do the interventions prove that attention weights are causal explanations?

No. They provide much stronger evidence than visualization alone, because changing a high-attention position perturbs the output far more than changing a low-attention position. However, the intervention is still a model intervention rather than a complete causal decomposition. Replacing a token can create an unusual or out-of-distribution context, and the network also contains residual connections, feed-forward layers, embeddings, and nonlinear interactions. Therefore the correct conclusion is that high-attention positions are **predictively important under these interventions**, not that the attention weights uniquely explain the model's reasoning.

## 5. Did the two selected examples generalize?

Only at the level of the broad claim that ambiguous predictions depend on context. The selected examples are illustrative and were chosen because the same merged symbol had different underlying values and was decoded correctly in both contexts. The 1,000-position intervention sample provides the aggregate evidence. It does **not** establish that the particular offsets highlighted in the two plots represent universal linguistic rules.

## 6. Did attention help more in the ambiguous experiment?

Not consistently. At 5% and 10% anchors, attention has a modest positive advantage over the matched MLP on ambiguous occurrences (about +0.97 and +1.19 percentage points). At 20% anchors, however, attention is worse by about 1.78 percentage points on ambiguous occurrences, and at 40% it is also slightly worse on that metric. In the bijective case the unanchored attention advantage is exactly zero at every anchor level.

Therefore the data do **not** support the general statement that attention is superior to a matched MLP. The intervention results instead support the more precise statement that the attention model has learned context-sensitive behavior for ambiguous symbols even though that behavior does not consistently translate into higher held-out decoding accuracy.

## 7. Contextual disambiguation is not unrevealed-name recovery

This distinction is essential. At the 20% anchor level in the ambiguous experiment, attention obtains

$$
7.34\%
$$

unanchored accuracy, while the matched MLP obtains

$$
8.15\%.
$$

Thus attention does not solve the naming problem for genuinely unrevealed syllables. Its strongest demonstrated benefit is sensitivity to context when the observed symbol is many-to-one and the possible values are represented in the permitted pseudo-label supervision.

## Final Step 9 critique

The LLM's narrow intervention predictions are supported: high-attention replacement and context permutation are much more disruptive than the low-attention control, especially for ambiguous occurrences. What is **not** supported is any stronger story that attention has discovered linguistic rules, that attention weights are a complete causal explanation, or that contextual attention overcomes the fundamental identifiability problem for unrevealed syllable names.

# 11. Failure-Mode Analysis


The final task examines five candidate limitations.

## 11.1 Too few anchors

This is directly tested by the 5%, 10%, 20%, and 40% experiments.

For the attention model, unanchored accuracy rises from

$$
0.03620
$$

at 5% anchors to

$$
0.15712
$$

at 40% anchors.

Thus too few revealed anchors are clearly an important limitation.

## 11.2 Too little text

The selected 20%-anchor attention architecture is retrained using deterministic prefixes containing 25%, 50%, and 100% of the training text.


| Training text | Training glyphs | Best epoch | Validation NLL | Whole-text test accuracy | Unanchored test accuracy |
| --- | --- | --- | --- | --- | --- |
| 25% | 13,520 | 7 | 0.539880 | 0.262525 | 0.093562 |
| 50% | 27,041 | 5 | 0.323335 | 0.265640 | 0.093562 |
| 100% | 54,082 | 4 | 0.195189 | 0.267976 | 0.093562 |

More text substantially reduces validation pseudo-label NLL, but it has only a small effect on true held-out whole-text accuracy.

Most importantly, unanchored accuracy remains

$$
0.093562
$$

for all three text amounts.

Therefore lack of raw training text is not the dominant explanation for the unrevealed-name problem in this experiment.

![Accuracy versus training text](../result%20images/hw6_2/task10_accuracy_vs_training_text.png)


## 11.3 Large or sparse vocabulary

| Training-frequency band | All evaluated occurrences | Mean training frequency | Whole-text accuracy | Unanchored occurrences | Unanchored accuracy |
| --- | --- | --- | --- | --- | --- |
| Q1 lowest frequency | 416 | 0.41 | 0.0841 | 340 | 0.0000 |
| Q2 | 126 | 2.00 | 0.1905 | 102 | 0.0000 |
| Q3 | 421 | 4.43 | 0.1900 | 341 | 0.0000 |
| Q4 highest frequency | 10594 | 466.27 | 0.2792 | 8505 | 0.1022 |

The association with frequency is strong.

The lowest-frequency quartile has whole-text accuracy only

$$
0.0841,
$$

while the highest-frequency quartile reaches

$$
0.2792.
$$

Among unanchored occurrences, the first three frequency bands have zero accuracy in the final 20%-anchor attention model, while the highest-frequency band reaches approximately

$$
0.1022.
$$

This is evidence of severe data sparsity for rare glyph types.

It does **not** prove that the 3,052-token vocabulary itself is causally too large, because frequency, linguistic content, and anchor availability are confounded.

![Accuracy by glyph frequency](../result%20images/hw6_2/task10_accuracy_by_glyph_frequency.png)


## 11.4 Model capacity

The architecture sweep gives a moderate negative Spearman association between parameter count and validation NLL, but the relationship is not monotone.

The 96-dimensional model is better than the 64-dimensional reference, yet:

- increasing context radius does not consistently help;
- four heads are not better than two;
- two layers are not better than one.

Thus there is some evidence that the smallest representation is underpowered, but the results do not support the simple claim that making the network larger will solve the task.

![Model capacity versus validation NLL](../result%20images/hw6_2/task10_model_capacity_vs_validation_nll.png)

## 11.5 Attention weights and useful context

This is directly tested by the Step 9 interventions.

For ambiguous centers, high-attention replacement and context permutation cause very large changes relative to the low-attention control.

Therefore the selected attention model **does** use useful surrounding context in the controlled ambiguity experiment.

The remaining decoding failures cannot be explained by saying that attention simply ignores context.

# 12. Interpretation Boundaries

The final results support several conclusions, but they also impose strict limits.

## Supported

- Increasing anchor coverage substantially improves hidden-label recovery.
- The selected attention model learns context-sensitive behavior for deliberately ambiguous glyphs.
- Ambiguous centers use broader contextual attention on average than unambiguous centers.
- High-attention context positions are much more influential than low-attention controls under the tested interventions.
- Context permutation strongly disrupts ambiguous predictions.
- Rare glyph types are substantially harder than frequent glyph types.
- More raw training text alone does not solve the unrevealed-name problem.

## Not supported

The experiment does **not** justify saying that:

- attention is uniformly more accurate than the matched MLP;
- attention discovers grammatical rules;
- attention weights are a complete causal explanation;
- the model reconstructs arbitrary unrevealed syllable names from context alone;
- a larger model would necessarily solve the remaining problem;
- the vocabulary size itself is proven to be causally responsible for low accuracy.

The key distinction is:

**Contextual disambiguation is not the same as unrevealed-name identification.**

Attention can distinguish different occurrences of a many-to-one observed symbol when the supervision provides the relevant candidate labels, while still failing to identify the names of genuinely unrevealed source syllables.

# 13. Reproducibility and Submitted Files

The final folder contains the complete executable workflow.

## `codes/`

- `HW6_2_Step1_Reuse_HW5_1_Corpus_Tokens_Glyphs_Map.ipynb`
- `HW6_2_Step2_Splits_Anchors_Information_Regime.ipynb`
- `HW6_2_Steps3-6_Attention_Decoder_Training_Comparison_Architecture.ipynb`
- `HW6_2_Steps7-8_Ambiguous_Glyphs_and_Attention_Analysis.ipynb`
- `HW6_2_Step9_LLM_Attention_Intervention_Critic.ipynb`
- `HW6_2_Step10_Failure_Modes.ipynb`

## `data/hw6_2/`

The final data folder includes:

- carried-over HW5.1 splits, anchors, source vocabulary, pseudo-label maps, and evaluation-only hidden map;
- selected architecture metadata;
- final bijective attention and MLP checkpoints;
- ambiguous attention and MLP checkpoints;
- prediction-level CSVs;
- relation-recovery tables;
- attention aggregate statistics;
- Step 9 intervention results;
- Step 10 failure-mode ablations.

## `result images/hw6_2/`

The final figures include:

- bijective accuracy and map-recovery curves;
- architecture validation-loss comparison;
- four Task 8 attention figures;
- Step 9 intervention-sensitivity figure;
- Step 10 training-text, frequency, and model-capacity figures.

The report uses only results contained in this final folder.

# Overall Conclusions

Homework 6.2 separates two different questions that can easily be confused:

1. **Can attention use context?**
2. **Does attention solve the decipherment problem better?**

The answer to the first question is clearly **yes** in the controlled ambiguous-glyph experiment.

The attention weights differ across contexts, ambiguous centers attend farther from the center on average, and direct interventions show that changing high-attention context positions can dramatically change the model output.

The answer to the second question is much more limited.

In the original bijective cipher, attention and the matched MLP have identical unanchored accuracy at every anchor level. In the ambiguous cipher, attention is slightly better at low anchor percentages but worse on several metrics at 20% and 40%.

Thus the correct final interpretation is:

> **Self-attention learns genuine context-sensitive disambiguation behavior, but contextual sensitivity is not the same as identifying source syllable names that were never revealed by anchors or the permitted reference-model supervision.**

The strongest limitations are therefore not simply architectural. They arise from the amount of identifying information available for rare and unrevealed glyph types.